# Quick start · Your first model call in 5 minutes

Deploy an Azure AI Gateway with a Microsoft Foundry model behind it, then call the model the way an app does. This
notebook is the Python track of the [quick start](README.md): it deploys the same [infra/main.bicep](infra/main.bicep)
as the CLI track, in one deployment, then makes the first call and checks that the gateway rejects a call without a
key.

**Deploy everything → Read the key → Call → Verify**

## The five building blocks

| Building block | In this notebook |
| --- | --- |
| **Gateway**: one runtime URL for your models and tools | The shared labs gateway, `aigw-labs-<6 characters>` |
| **Model provider**: the backend, reached with the gateway's managed identity | `foundry-models`, for a new Foundry account |
| **Model**: a backend deployment | The Foundry deployment `gpt-6-luna` |
| **Alias**: the name apps send in `model` | `gpt-6-luna` |
| **Runtime API key**: identifies the calling app | `app-first-call` |

The notebook uses the same names as notebooks 01 and 02, so the later notebooks work on this gateway.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize the lab

`utils.Lab` reads your `az login` session and the shared defaults: resource group `rg-ai-gateway-labs` in `eastus2`,
and a gateway and Foundry account named after your subscription. Override them with the `AIGW_*` environment
variables described in [shared/utils.py](../shared/utils.py) before you start the kernel. The quick start creates its
own Foundry account, so it doesn't use `AIGW_FOUNDRY_ACCOUNT`.

In [ ]:
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

lab = utils.Lab("quickstart", needs_gateway=False)
assert not lab.own_foundry_account and lab.foundry_resource_group == lab.resource_group, \
    "The quick start creates its own Foundry account. Unset AIGW_FOUNDRY_ACCOUNT, or run notebooks 01 and 02 instead."
print(f"🏭 Foundry:        {lab.foundry_account}")

### 1. Deploy the gateway and a model

One Bicep deployment creates the gateway, a Foundry account with a `gpt-6-luna` deployment, the model provider, the
alias, and the runtime key. It reuses the Bicep modules of labs 01 and 02, and takes about three minutes. Running the
cell again updates the same resources.

In [ ]:
lab.register_providers("Microsoft.ApiManagement", "Microsoft.CognitiveServices")
lab.create_resource_group()

out = lab.deploy("infra/main.bicep", gatewayName=lab.gateway_name, foundryAccountName=lab.foundry_account,
                 location=lab.location)
for name, value in out.items():
    print(f"📤 {name:<20} {value}")

### 2. Read the runtime API key

Deployment outputs aren't secret, so the template doesn't return the key's value. `lab.api_key` reads it with
`listSecrets`. The value stays in memory: the notebook never prints it or writes it to a file.

In [ ]:
api_key, _ = lab.api_key(out["apiKeyName"])
print(f"🔑 Read key {out['apiKeyName']} ({len(api_key)} characters, not shown)")

### 3. Make your first call

Point the OpenAI client at the `modelEndpoint` output, `<gateway URL>/default/models/openai/v1`, and pass the
runtime key as `api_key`. The `model` field holds the alias. A new role assignment or alias can take a minute to
apply, so `utils.retry` retries `401`, `403`, and `404` responses.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=out["modelEndpoint"], api_key=api_key)
chat = utils.retry(lambda: client.chat.completions.create(
    model=out["aliasName"], messages=[{"role": "user", "content": "In one sentence, what does an AI gateway do?"}]))

print(f"📨 {chat.model}: {chat.choices[0].message.content}")
print(f"🔢 Tokens: {chat.usage.prompt_tokens} in, {chat.usage.completion_tokens} out")

### 4. Verify

The gateway answers only callers that it can identify: the same request returns `200` with the key and `401`
without it.

In [ ]:
import requests

chat_url = f"{out['modelEndpoint']}/chat/completions"
payload  = {"model": out["aliasName"], "messages": [{"role": "user", "content": "Hello"}]}

with_key = requests.post(chat_url, json=payload, headers={"api-key": api_key}, timeout=60)
print(f"📨 With key:      HTTP {with_key.status_code}")
assert with_key.status_code == 200, with_key.text[:500]

without_key = requests.post(chat_url, json=payload, timeout=60)
print(f"📨 Without key:   HTTP {without_key.status_code}")
assert without_key.status_code == 401
print(f"✅ Your gateway is live. Open https://ai.gateway.azure.com and select {out['gatewayName']} to see it.")

### 5. Clean up

Keep the gateway to continue with the other labs: their notebooks find it with the same defaults. When you're done,
uncomment the lines below. `lab.delete_foundry_account` deletes and purges the Foundry account, which frees its name
and stops any charges.

In [ ]:
# Uncomment to delete the gateway and the Foundry account. The other lab notebooks reuse them.
# lab.delete(lab.gateway_id, wait=True)
# lab.delete_foundry_account()
print(f"ℹ️ Kept {lab.gateway_name} and {lab.foundry_account} in {lab.resource_group}")

## Troubleshooting

- **`AuthorizationFailed` on `roleAssignments` in step 1**: the template gives the gateway's managed identity the
  Foundry User role, so you need **Owner**, or **Role Based Access Control Administrator**, on the resource group.
- **`InsufficientQuota` in step 1**: there's no `gpt-6-luna` capacity left in the region. Set `AIGW_LOCATION` to
  `swedencentral` and restart the kernel.
- **`FlagMustBeSetForRestore` in step 1**: a Foundry account with the same name is soft-deleted. Purge it with
  `az cognitiveservices account purge`, then rerun the cell.
- **`401` on every call**: send the key in `api-key` *or* `Authorization: Bearer`, not both (`400`).

## Next steps

- Take the tour in [How Azure AI Gateway works](../00-how-ai-gateway-works/README.md), with a 3-minute video.
- Go step by step with [notebook 01](../01-provision-gateway/01-provision-gateway.ipynb) and [notebook 02](../02-first-model-call/02-first-model-call.ipynb): they reuse this gateway and model.
- Put models from several providers behind stable names in [lab 03](../03-multi-provider-aliases/README.md).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/quickstart/](https://azure.github.io/ai-gateway/labs/quickstart/)